# Fase 1 — El ejemplo TDIP de pyGIMLi, paso a paso
Notebook de práctica del Laboratorio 3. Corre cada celda en orden (**Shift + Enter**) y antes de correrla intenta predecir qué va a salir.
El documento `Estudio/02_Fase1_Script_pyGIMLi.pdf` explica cada resultado.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import pygimli as pg
import pygimli.meshtools as mt
from pygimli.physics import ert

pg.setLogLevel(30)          # solo advertencias
np.random.seed(1234)        # ruido reproducible
print("pyGIMLi", pg.__version__)

## 1. Geometría y malla
Tres capas (interfaces a 1 m y 5 m) y un círculo de radio 1 m a 3 m de profundidad. 21 electrodos cada 1 m, arreglo dipolo-dipolo.

In [ ]:
CIRC_POS, CIRC_R = (0.0, -3.0), 1.0

world = mt.createWorld(start=[-50, 0], end=[50, -50], layers=[-1, -5], worldMarker=True)
scheme = ert.createData(elecs=pg.utils.grange(start=-10, end=10, n=21), schemeName="dd")
circle = mt.createCircle(pos=list(CIRC_POS), radius=CIRC_R, marker=4)
world += circle
for pos in scheme.sensorPositions():
    world.createNode(pos)
    world.createNode(pos + [0.0, -0.1])
mesh = mt.createMesh(world, quality=34.4)

print(f"{scheme.size()} cuadripolos, {mesh.cellCount()} celdas")
ax, cb = pg.show(mesh, markers=True, showMesh=True, boundaryMarkers=False)
ax.plot(pg.x(scheme), pg.y(scheme), "mo", ms=4)
ax.set_xlim(-15, 15); ax.set_ylim(-10, 0.5);

## 2. Simulación en dominio del tiempo
Los arreglos se indexan **por marcador** (la posición 0 no se usa). El círculo tiene la misma ρ que su capa, pero m = 0.1.

⚠️ Pasamos una **copia** del `scheme`: `simulate` escribe los resultados dentro del objeto que recibe.

In [ ]:
res = np.array([0, 100, 50, 10, 50.])
m   = np.array([0, 0, 0, 0, 0.1])

data = ert.ERTIPManager().simulate(mesh=mesh, scheme=pg.DataContainerERT(scheme), res=res, m=m)

fig, axs = plt.subplots(1, 2, figsize=(11, 3.4))
ert.show(data, "rhoa", ax=axs[0], label=r"$\rho_a$ ($\Omega$m)", cMap="Spectral_r")
ert.show(data, "ip", ax=axs[1], label="$m_a$ (mV/V)", logScale=False, cMap="magma_r")
axs[0].set_title("Resistividad aparente"); axs[1].set_title("Cargabilidad aparente")
print(f"m_a máxima = {max(data['ip']):.2f} mV/V  (el círculo tiene 100 mV/V)")

## 3. Inversión como en el ejemplo original
Primero DC (λ = 20), después IP (λ = 100 por defecto). Fíjate en los dos χ².

In [ ]:
def m_in_circle(mgr):
    c = np.array(mgr.paraDomain.cellCenters())
    inside = np.hypot(c[:, 0] - CIRC_POS[0], c[:, 1] - CIRC_POS[1]) < CIRC_R
    mod = np.asarray(mgr.modelIP)
    return mod[inside].mean() * 1000, mod.max() * 1000

def show_inversion(mgr, titulo=""):
    fig, axs = plt.subplots(1, 2, figsize=(11, 3.2))
    mgr.showResult(ax=axs[0], cMap="Spectral_r")
    pg.viewer.mpl.drawPLC(axs[0], circle, fitView=False, fillRegion=False, color="k")
    axs[0].set_title(f"ρ invertida (χ² = {mgr.inv.chi2():.2f})")
    mgr.showIPModel(ax=axs[1])
    pg.viewer.mpl.drawPLC(axs[1], circle, fitView=False, fillRegion=False, color="white")
    axs[1].set_title(f"m invertida (χ² = {mgr.invIP.chi2():.2f}) {titulo}")
    for a in axs:
        a.set_xlim(-12, 12); a.set_ylim(-8, 0)

data["err"] = 0.03
mgr = ert.ERTIPManager(data)
mgr.invert(zWeight=0.2, quality=34.4, verbose=False)
show_inversion(mgr)
print("m en el círculo: media = %.1f, máx = %.1f mV/V" % m_in_circle(mgr))

## 4. Verificación 1 — semiespacio homogéneo
Si todo el subsuelo es igual, cualquier medición debe dar exactamente ρa = ρ y m_a = m.

In [ ]:
dhs = ert.ERTIPManager().simulate(mesh=mesh, scheme=pg.DataContainerERT(scheme),
                                  res=np.array([0, 50, 50, 50, 50.]),
                                  m=np.array([0, 0.1, 0.1, 0.1, 0.1]))
print(f"rho_a: {min(dhs['rhoa']):.2f} – {max(dhs['rhoa']):.2f} Ohm·m   (esperado 50)")
print(f"m_a  : {min(dhs['ip']):.2f} – {max(dhs['ip']):.2f} mV/V      (esperado 100)")

## 5. Verificación 2 — frecuencia vs tiempo
El cuerpo debería "diluirse" en la misma proporción en los dos dominios.

In [ ]:
rhomap = [[1, 100. + 0j], [2, 50. + 0j], [3, 10. + 0j], [4, 50. + 1j]]
dfd = ert.simulate(mesh, res=rhomap, scheme=pg.DataContainerERT(scheme), verbose=False)
phi_true = np.arctan(1 / 50) * 1000
phi_app = np.abs(np.array(dfd["phia"])).max() * 1000
print(f"FD: {phi_app:.2f} de {phi_true:.1f} mrad -> {phi_app / phi_true * 100:.1f} %")
print(f"TD: {max(data['ip']):.2f} de 100 mV/V -> {max(data['ip']):.1f} %")

## 6. Verificación 3 — ¿qué λ da χ² ≈ 1?
Cambia la lista `lams` o `absErr` y vuelve a correr. ¿Qué pasa si pones `absErr = 0.0001`?

In [ ]:
lams, absErr = (100, 20, 10, 5, 1), 0.001
filas = []
for lam in lams:
    g = ert.ERTIPManager(data)
    g.invertDC(zWeight=0.2, quality=34.4, verbose=False)
    g.invertTDIP(lam=lam, absoluteError=absErr, zWeight=0.2, verbose=False)
    media, maximo = m_in_circle(g)
    filas.append((lam, g.invIP.chi2(), media, maximo))
    print(f"λ = {lam:>4}   χ² = {filas[-1][1]:5.2f}   m media = {media:5.1f}   m máx = {maximo:5.1f} mV/V")

fig, (a1, a2) = plt.subplots(1, 2, figsize=(11, 3.2))
a1.plot([f[0] for f in filas], [f[1] for f in filas], "o-"); a1.axhline(1, ls="--", c="g")
a1.set_ylabel("χ² IP"); a1.set_yscale("log")
a2.plot([f[0] for f in filas], [f[2] for f in filas], "o-", label="media")
a2.plot([f[0] for f in filas], [f[3] for f in filas], "s:", label="máx")
a2.axhline(100, ls="--", c="r"); a2.set_ylabel("m en el círculo (mV/V)"); a2.legend()
for a in (a1, a2):
    a.set_xscale("log"); a.invert_xaxis(); a.set_xlabel("λ")

## 7. Verificación 4 — datos con ruido realista
3 % de ruido en ρa y 0.5 mV/V en la cargabilidad. Prueba otros valores de `lam_ip`.

In [ ]:
dn = ert.ERTIPManager().simulate(mesh=mesh, scheme=pg.DataContainerERT(scheme), res=res, m=m)
dn["rhoa"] = np.array(dn["rhoa"]) * (1 + 0.03 * np.random.randn(dn.size()))
dn["ip"] = np.array(dn["ip"]) + 0.5 * np.random.randn(dn.size())
dn["err"] = 0.03

lam_ip = 20
gn = ert.ERTIPManager(dn)
gn.invertDC(zWeight=0.2, quality=34.4, verbose=False)
gn.invertTDIP(lam=lam_ip, absoluteError=0.0005, zWeight=0.2, verbose=False)
show_inversion(gn, "(con ruido)")
print("m en el círculo: media = %.1f, máx = %.1f mV/V" % m_in_circle(gn))